# Final LSTM tuning: Real-Time Flood Data Pipeline

This notebook runs the last tuning round for the event-weighted LSTM, from a clean session each time.
Run the cells in order with **Runtime > Run all**. Use a GPU runtime (Runtime > Change runtime type > T4 GPU).

**Plan, fixed before any result is seen**

1. Data: `features_daily_audited.csv`. Development folds 2015-18 → 2019, 2015-20 → 2021-22, 2015-22 → 2023.
   Final test 2015-23 → 2024-25. Unknown days are never trained on or scored.
2. Search (full model, on the development folds only): learning rate {0.0003, 0.001} × window {7, 14} ×
   architecture {32 units with dropout 0.3, 64 units with dropout 0.5}, each scored after every epoch from 1 to 15.
   Event weights and batch size 64 as in the deployed model. Eight seeds per setting, scores averaged.
3. Choice: the setting and epoch with the highest **pooled development average precision** of the eight-seed mean.
   Ties (within 0.001) go to fewer epochs, then the lower learning rate, then the shorter window, then 32 units.
4. Threshold: best day-level F1 on the pooled development folds (candidates are development-score quantiles 0.90 to
   0.999, ties to the higher threshold), the same rule as before. An alarm-rate threshold is reported beside it as a
   sensitivity check only and is never used to choose.
5. Provisional model (no CHIRPS): the chosen window, learning rate and architecture; only its epochs and threshold
   are chosen the same way on the development folds.
6. The final test cell is run **once**. Its result is reported whatever it is.

In [ ]:
# 1. Setup
import os, json, time, zipfile
os.environ["TF_DETERMINISTIC_OPS"] = "1"
import numpy as np, pandas as pd, tensorflow as tf
tf.get_logger().setLevel("ERROR")
from numpy.lib.stride_tricks import sliding_window_view
print("TensorFlow", tf.__version__, "| Keras", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "")
print("GPU:", tf.config.list_physical_devices("GPU") or "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
# 2. Data: upload features_daily_audited.csv from
#    data_collection/data/08_model_ready/ in the project folder, or point DATA_PATH at a copy in Google Drive.
from google.colab import drive, files
DATA_PATH = "/content/drive/MyDrive/flood_tuning/features_daily_audited.csv"   # optional Drive copy
OUT_DIR = "/content/drive/MyDrive/flood_tuning/results"                       # results saved here if Drive is used
try:
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted:", e)
if not os.path.exists(DATA_PATH):
    print("Upload features_daily_audited.csv")
    up = files.upload()
    DATA_PATH = "/content/" + next(iter(up))
    OUT_DIR = "/content/results"
os.makedirs(OUT_DIR, exist_ok=True)
RAW = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values(["catchment", "date"])
print(len(RAW), "rows,", RAW["date"].min().date(), "to", RAW["date"].max().date())
print(RAW["label_status"].value_counts().to_dict())

In [ ]:
# 3. Shared code, copied from the repository (sequences.py, save_final_model.py, event_train_evaluate.py)
SEQ_FEATS = ["rain_mm", "imerg_max_rate", "imerg_total_mm", "imerg_hours_ge5", "era5_rain_mm", "soil_1", "soil_2",
             "temp_c", "dewpoint_depression", "pressure_hpa", "month_sin", "month_cos"]
CATCHMENTS = ["ngong", "mathare", "nairobi"]
DEV_FOLDS = [(range(2015, 2019), [2019]), (range(2015, 2021), [2021, 2022]), (range(2015, 2023), [2023])]
FINAL_FOLD = (range(2015, 2024), [2024, 2025])
SEEDS = list(range(8))
DAY = pd.Timedelta(days=1)
QS = np.round(np.arange(0.90, 0.9991, 0.001), 3)
_windows = {}


def make_windows(t, drop=()):
    # one window per catchment-day: the t days ending on that day, labelled with the last day
    if (t, drop) not in _windows:
        df = RAW.copy()
        for c in CATCHMENTS:
            df[f"is_{c}"] = (df["catchment"] == c).astype("float32")
        feats = [f for f in SEQ_FEATS + [f"is_{c}" for c in CATCHMENTS] if f not in drop]
        xs, metas = [], []
        for c, g in df.groupby("catchment"):
            vals = g[feats].values.astype("float32")
            xs.append(sliding_window_view(vals, (t, vals.shape[1]))[:, 0])
            metas.append(g.iloc[t - 1:][["catchment", "date", "label_status", "event_id"]].copy())
        meta = pd.concat(metas).reset_index(drop=True)
        meta["year"] = meta["date"].dt.year
        _windows[(t, drop)] = (np.concatenate(xs), meta, feats)
    return _windows[(t, drop)]


def make_model(n_feats, t, units, dropout, lr):
    m = tf.keras.Sequential([
        tf.keras.layers.Input((t, n_feats)),
        tf.keras.layers.LSTM(units),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="binary_crossentropy")
    return m


def fold_data(X, meta, train_years, val_years):
    st = meta["label_status"].values
    flood, known = st == "flood", np.isin(st, ["flood", "no_flood"])
    tr = meta["year"].isin(list(train_years)).values & known
    va = meta["year"].isin(list(val_years)).values
    flat = X[tr].reshape(-1, X.shape[2])
    mu, sd = flat.mean(0), flat.std(0) + 1e-6
    w = float(np.sqrt((tr & ~flood).sum() / (tr & flood).sum()))          # class weight
    eid = meta["event_id"].where(flood)
    L = meta[tr & flood].groupby(eid[tr & flood]).size()                   # days per event
    sw = np.where(flood, w * L.mean() / eid.map(L).fillna(1).values, 1.0)  # event weights
    return (X[tr] - mu) / sd, flood[tr].astype("float32"), sw[tr], (X[va] - mu) / sd, va, mu, sd


def run_setting(t, units, dropout, lr, max_ep, folds, drop=(), keep_models=False, log=None):
    # eight seeds; scores after every epoch, averaged over seeds -> {epoch: DataFrame(val, catchment, date, score)}
    X, meta, feats = make_windows(t, drop)
    out, models = {}, []
    for train_years, val_years in folds:
        Xtr, ytr, sw, Xva, va, mu, sd = fold_data(X, meta, train_years, val_years)
        S = np.zeros((max_ep, len(Xva)))
        for seed in SEEDS:
            tf.keras.utils.set_random_seed(seed)
            m = make_model(X.shape[2], t, units, dropout, lr)
            for ep in range(max_ep):
                h = m.fit(Xtr, ytr, sample_weight=sw, epochs=1, batch_size=64, verbose=0)
                S[ep] += m.predict(Xva, batch_size=1024, verbose=0).ravel() / len(SEEDS)
                if log is not None:
                    log.append({"val": "-".join(map(str, val_years)), "seed": seed, "epoch": ep + 1,
                                "loss": h.history["loss"][0]})
            if keep_models:
                models.append(m)
        base = meta.loc[va, ["catchment", "date"]].reset_index(drop=True)
        for ep in range(max_ep):
            d = base.copy(); d["val"] = "-".join(map(str, val_years)); d["score"] = S[ep]
            out.setdefault(ep + 1, []).append(d)
    out = {ep: pd.concat(v, ignore_index=True) for ep, v in out.items()}
    return (out, models, feats, mu, sd) if keep_models else out


LAB = RAW[["catchment", "date", "label_status", "event_id"]]


def prepare(d):
    m = d.merge(LAB, on=["catchment", "date"])
    parts = []
    for _, g in m.groupby("val"):
        ev = (g[g["label_status"] == "flood"].groupby("event_id")
              .agg(catchment=("catchment", "first"), start=("date", "min"), end=("date", "max"), days=("date", "size")))
        known = g[g["label_status"].isin(["flood", "no_flood"])]
        parts.append((g, ev, known["label_status"].eq("flood").values, known["score"].values))
    return parts


def ap(y, s):
    o = np.argsort(-s, kind="mergesort")
    y, s = y[o], s[o]
    last = np.r_[np.diff(s) != 0, True]
    tp = np.cumsum(y)[last]
    return float(np.sum(np.diff(np.r_[0, tp / tp[-1]]) * tp / np.arange(1, len(y) + 1)[last]))


def pooled_ap(parts):
    return ap(np.concatenate([p[2] for p in parts]), np.concatenate([p[3] for p in parts]))


def stats(parts, thr, cy):
    n = det = det1 = n1 = eps = false_eps = tp = fp = fn = 0
    for g, ev, y, s in parts:
        a = s >= thr
        tp += int((a & y).sum()); fp += int((a & ~y).sum()); fn += int((~a & y).sum())
        alarms = g[g["score"] >= thr]
        for e in ev.itertuples():
            x = alarms[alarms["catchment"] == e.catchment]
            hit = bool(((x["date"] >= e.start - DAY) & (x["date"] <= e.end + DAY)).any())
            det += hit
            if e.days == 1:
                n1 += 1; det1 += hit
        n += len(ev)
        for c, x in alarms.sort_values("date").groupby("catchment"):
            evc = ev[ev["catchment"] == c]
            for _, ep in x.groupby((x["date"].diff() != DAY).cumsum()):
                eps += 1
                false_eps += not ((evc["start"] - DAY <= ep["date"].max()) & (evc["end"] + DAY >= ep["date"].min())).any()
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn)
    er = det / n
    ep_p = (eps - false_eps) / eps if eps else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0,
            "events": n, "detected": det, "one_day_detected": f"{det1}/{n1}", "multi_day_detected": f"{det - det1}/{n - n1}",
            "event_recall": er, "episodes": eps, "false_episodes": false_eps, "episode_precision": ep_p,
            "event_f1": 2 * ep_p * er / (ep_p + er) if ep_p + er else 0.0, "false_eps_per_cy": false_eps / cy}


def pick_threshold(dev_scores):
    dp = prepare(dev_scores)
    grid = np.unique(np.quantile(dev_scores["score"], QS))
    sw = pd.DataFrame([{"threshold": t, **stats(dp, t, 12)} for t in grid])
    return sw.sort_values(["f1", "threshold"], ascending=[False, False]).iloc[0]


def wilson(k, n, z=1.96):
    p = k / n; c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return round(c - h, 2), round(c + h, 2)

print("ready")

In [ ]:
# 4. Development search (about an hour on a T4). Results are saved after each setting, so a rerun skips them.
GRID = [{"lr": lr, "window": t, "units": u, "dropout": d}
        for lr in [0.0003, 0.001] for t in [7, 14] for u, d in [(32, 0.3), (64, 0.5)]]
MAX_EP = 15
curve_file = f"{OUT_DIR}/dev_curve.csv"
curve = pd.read_csv(curve_file) if os.path.exists(curve_file) else pd.DataFrame()
for g in GRID:
    key = f"lr{g['lr']}_w{g['window']}_u{g['units']}_d{g['dropout']}"
    if len(curve) and key in set(curve["setting"]):
        continue
    t0 = time.time()
    scores = run_setting(g["window"], g["units"], g["dropout"], g["lr"], MAX_EP, DEV_FOLDS)
    rows = [{"setting": key, **g, "epoch": ep, "dev_ap": pooled_ap(prepare(s))} for ep, s in scores.items()]
    curve = pd.concat([curve, pd.DataFrame(rows)], ignore_index=True)
    curve.to_csv(curve_file, index=False)
    best = max(rows, key=lambda r: r["dev_ap"])
    print(f"{key}: best epoch {best['epoch']}, dev AP {best['dev_ap']:.3f}  ({time.time() - t0:.0f}s)")

In [ ]:
# 5. Choose the setting and epochs (pre-stated rule), then the threshold
c = curve.copy()
top = c["dev_ap"].max()
c["tie"] = c["dev_ap"] >= top - 0.001
choice = c[c["tie"]].sort_values(["epoch", "lr", "window", "units"]).iloc[0]
print(f"Chosen: lr {choice.lr}, window {int(choice.window)}, units {int(choice.units)}, dropout {choice.dropout}, "
      f"epochs {int(choice.epoch)}, pooled dev AP {choice.dev_ap:.3f} (best {top:.3f})")
print(c.pivot_table(index="setting", columns="epoch", values="dev_ap").round(3).to_string())

CH = {"lr": float(choice.lr), "window": int(choice.window), "units": int(choice.units),
      "dropout": float(choice.dropout), "epochs": int(choice.epoch)}
dev_full = run_setting(CH["window"], CH["units"], CH["dropout"], CH["lr"], CH["epochs"], DEV_FOLDS)[CH["epochs"]]
thr_full = pick_threshold(dev_full)
dev_rate_full = float((dev_full["score"] >= thr_full["threshold"]).mean())
print(f"Full model threshold {thr_full['threshold']:.4f}: dev F1 {thr_full['f1']:.3f}, dev event recall "
      f"{thr_full['event_recall']:.3f}, dev alarm rate {dev_rate_full:.3%}")

In [ ]:
# 6. Provisional model (no CHIRPS rain_mm): same window, learning rate and architecture; epochs and threshold from dev
fast = run_setting(CH["window"], CH["units"], CH["dropout"], CH["lr"], MAX_EP, DEV_FOLDS, drop=("rain_mm",))
fast_curve = pd.DataFrame([{"epoch": ep, "dev_ap": pooled_ap(prepare(s))} for ep, s in fast.items()])
ftop = fast_curve["dev_ap"].max()
FAST_EP = int(fast_curve[fast_curve["dev_ap"] >= ftop - 0.001]["epoch"].min())
thr_fast = pick_threshold(fast[FAST_EP])
dev_rate_fast = float((fast[FAST_EP]["score"] >= thr_fast["threshold"]).mean())
fast_curve.to_csv(f"{OUT_DIR}/dev_curve_fast.csv", index=False)
print(fast_curve.round(3).T.to_string())
print(f"Provisional: epochs {FAST_EP}, threshold {thr_fast['threshold']:.4f}, dev F1 {thr_fast['f1']:.3f}")
json.dump({"full": {**CH, "threshold": float(thr_full["threshold"]), "dev_alarm_rate": dev_rate_full},
           "fast": {**CH, "epochs": FAST_EP, "threshold": float(thr_fast["threshold"]), "dev_alarm_rate": dev_rate_fast}},
          open(f"{OUT_DIR}/choice.json", "w"), indent=2)
print("choice saved:", json.load(open(f"{OUT_DIR}/choice.json")))

In [ ]:
# 7. Loss curve for the report: the chosen setting on the 2023 fold, train and validation loss per epoch (4 seeds)
X, meta, _ = make_windows(CH["window"])
Xtr, ytr, sw, Xva, va, _, _ = fold_data(X, meta, *DEV_FOLDS[2])
st = meta["label_status"].values
known_va = np.isin(st[va], ["flood", "no_flood"])
yva = (st[va] == "flood").astype("float32")
eid = meta["event_id"].where(st == "flood")
L = meta[va & (st == "flood")].groupby(eid[va & (st == "flood")]).size()
w = float(np.sqrt((~(st == "flood") & np.isin(st, ["flood", "no_flood"]) & meta["year"].isin(list(DEV_FOLDS[2][0])).values).sum()
                  / ((st == "flood") & meta["year"].isin(list(DEV_FOLDS[2][0])).values).sum()))
swva = np.where(yva == 1, w * L.mean() / eid[va].map(L).fillna(1).values, 1.0)
hist = []
for seed in range(4):
    tf.keras.utils.set_random_seed(seed)
    m = make_model(X.shape[2], CH["window"], CH["units"], CH["dropout"], CH["lr"])
    h = m.fit(Xtr, ytr, sample_weight=sw, epochs=MAX_EP, batch_size=64, verbose=0,
              validation_data=(Xva[known_va], yva[known_va], swva[known_va]))
    hist.append([h.history["loss"], h.history["val_loss"]])
hist = np.array(hist).mean(0)
pd.DataFrame({"epoch": range(1, MAX_EP + 1), "train_loss": hist[0], "val_loss": hist[1]}).to_csv(
    f"{OUT_DIR}/loss_curve.csv", index=False)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
ax.plot(range(1, MAX_EP + 1), hist[0], color="#2a78d6", lw=2, label="Training loss (2015-2022)")
ax.plot(range(1, MAX_EP + 1), hist[1], color="#eb6834", lw=2, label="Validation loss (2023)")
ax.axvline(CH["epochs"], color="#52514e", ls="--", lw=1); ax.text(CH["epochs"] + 0.2, ax.get_ylim()[1] * 0.9, "chosen epochs", fontsize=9)
ax.set_xlabel("Epoch"); ax.set_ylabel("Weighted binary cross-entropy"); ax.legend(frameon=False)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/loss_curve.png"); plt.show()

## Final test: run this cell once

Everything above used the development years only. This trains on 2015-2023 with the chosen settings and scores
2024-2025. Do not change anything above after running it.

In [ ]:
# 8. FINAL TEST (once) and saving the models in the format the backend loads
ch = json.load(open(f"{OUT_DIR}/choice.json"))
results, saved = {}, {}
for kind, drop in [("full", ()), ("fast", ("rain_mm",))]:
    c = ch[kind]
    scores, models, feats, mu, sd = run_setting(c["window"], c["units"], c["dropout"], c["lr"], c["epochs"],
                                                [FINAL_FOLD], drop=drop, keep_models=True)
    s = scores[c["epochs"]]
    s.to_csv(f"{OUT_DIR}/final_scores_{kind}.csv", index=False)
    parts = prepare(s)
    r = {"threshold": c["threshold"], "ap": pooled_ap(parts), **stats(parts, c["threshold"], 6)}
    r["event_recall_ci"] = wilson(r["detected"], r["events"])
    r["precision_ci"] = wilson(r["tp"], r["tp"] + r["fp"]); r["recall_ci"] = wilson(r["tp"], r["tp"] + r["fn"])
    # sensitivity only: the threshold that gives the development alarm rate on the test years
    t_rate = float(np.quantile(s["score"], 1 - c["dev_alarm_rate"]))
    r["sensitivity_alarm_rate"] = {"threshold": t_rate, **{k: v for k, v in stats(parts, t_rate, 6).items()
                                   if k in ("precision", "recall", "f1", "detected", "event_f1", "false_eps_per_cy")}}
    results[kind] = r
    name, version = ("lstm_raw", "v3") if kind == "full" else ("lstm_fast", "v2")
    d = f"/content/models/flood_lstm{'' if kind == 'full' else '_fast'}"
    os.makedirs(d, exist_ok=True)
    for seed, m in zip(SEEDS, models):
        m.save_weights(f"{d}/lstm_raw_seed{seed}.weights.h5")
    json.dump({"name": name, "version": version, "training": "event-weighted", "features": feats,
               "window": c["window"], "units": c["units"], "dropout": c["dropout"], "learning_rate": c["lr"],
               "epochs": c["epochs"], "mean": mu.tolist(), "std": sd.tolist(), "threshold": c["threshold"],
               "seeds": SEEDS, "train_years": [2015, 2023], "labels": "features_daily_audited.csv",
               "target": "today", "tuned_in": "colab/final_tuning.ipynb"}, open(f"{d}/config.json", "w"), indent=2)
    json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in r.items()},
              open(f"{d}/evaluation.json", "w"), indent=2, default=float)
json.dump(results, open(f"{OUT_DIR}/final_results.json", "w"), indent=2, default=float)
show = pd.DataFrame({k: {x: y for x, y in v.items() if x != "sensitivity_alarm_rate"} for k, v in results.items()})
print(show.to_string())
print("\nSensitivity (alarm-rate threshold, not used to choose):")
print(pd.DataFrame({k: v["sensitivity_alarm_rate"] for k, v in results.items()}).round(3).to_string())
with zipfile.ZipFile("/content/final_models.zip", "w") as z:
    for root, _, fs in os.walk("/content/models"):
        for f in fs:
            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), "/content"))
    for f in os.listdir(OUT_DIR):
        z.write(f"{OUT_DIR}/{f}", f"results/{f}")
files.download("/content/final_models.zip")

In [ ]:
# 9. Confusion matrix of the full model on 2024-2025 (labelled days only)
r = results["full"]
tn = int(sum(len(p[2]) for p in prepare(pd.read_csv(f"{OUT_DIR}/final_scores_full.csv", parse_dates=["date"])))) - r["tp"] - r["fp"] - r["fn"]
cm = pd.DataFrame([[r["tp"], r["fn"]], [r["fp"], tn]], index=["Actual flood", "Actual no flood"],
                  columns=["Predicted flood", "Predicted no flood"])
print(cm)
cm.to_csv(f"{OUT_DIR}/confusion_matrix.csv")